# NOG: Regression Model Path Generator


## Install libraries

This notebook uses five external libraries:

- **pandas** loads, filters, and organizes the data.
- **numpy** calculates returns, fits the regression, and generates Brownian increments.
- **matplotlib** plots the fitted model and simulated price paths.
- **pandas_market_calendars** supplies NYSE trading dates, including holidays.

- **pyarrow** writes and reads the Parquet path export.


In [ ]:
%pip install pandas numpy matplotlib pandas_market_calendars pyarrow


## Import libraries

`pandas` (`pd`) handles data tables, `numpy` (`np`) handles numerical calculations and random draws, and `matplotlib.pyplot` (`plt`) draws charts. `PercentFormatter` formats return axes as percentages. `pandas_market_calendars` (`mcal`) supplies the trading calendar.

`json` reads JSON files, and `Path` handles file locations. Both are included with Python.

`pyarrow` is used by pandas as the Parquet engine when exporting; it needs no separate import.


In [ ]:
import json
from pathlib import Path
from tempfile import NamedTemporaryFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas_market_calendars as mcal
from matplotlib.ticker import PercentFormatter


## Data and simulation setup

Historical observations cover **1 January 2020 through 31 December 2025**, inclusive. The **10,000 synthetic paths** cover **1 January–31 December 2026** with the current settings. Each daily column represents one NYSE trading session. `day_0` is the latest closing price before the simulation; the fixed random seed makes runs reproducible.

A joint regression uses all complete historical trading-day intervals to estimate daily background log return and the contributions of qualifying oil shocks and interest-rate changes. The scenario combines background growth across every simulation session with each specified shock contribution once. The resulting cumulative predicted log return determines the terminal **median**; the Gaussian variance correction determines the **expected mean**. Standard GBM generates individual paths with different endpoints.

Historical stock log returns supply daily volatility. These simulations use constant volatility and fixed estimated coefficients; they are scenario distributions, not validated forecast intervals. Run this notebook from the project folder containing `data`.


In [ ]:
TICKER = "NOG"
DATA_DIR = Path("data")
EXPORT_ROOT = Path("export")
DATA_START_DATE = pd.Timestamp("2020-01-01")
DATA_END_DATE = pd.Timestamp("2025-12-31")
SIMULATION_START_DATE = pd.Timestamp("2026-01-01")
SIMULATION_END_DATE = pd.Timestamp("2027-01-01")  # Exclusive.
N_PATHS = 10_000
RANDOM_SEED = 2026

if not DATA_DIR.is_dir():
    raise FileNotFoundError("Run this notebook from the project folder containing 'data'.")
if not DATA_START_DATE <= DATA_END_DATE < SIMULATION_START_DATE < SIMULATION_END_DATE:
    raise ValueError("Historical dates must precede a simulation period with a positive duration.")
if not isinstance(N_PATHS, int) or isinstance(N_PATHS, bool) or N_PATHS < 1:
    raise ValueError("N_PATHS must be a positive integer.")


### Predefined shock events

Each row defines an event's date, publisher, type, and relative change. Oil changes of `0.48` and `-0.29` mean +48% and -29%. A rate change of `0.07` means a **7% relative increase in the current reference rate**, converted to percentage points. Applied rate events update that reference sequentially.

Events are mapped to the next NYSE session on or after their date. Events outside the simulation, or with no remaining session, are recorded and skipped. Multiple events on one session contribute to the same pair of regression inputs; their combined daily oil move must satisfy the oil filter. The background intercept is included **once per trading session**, independently of the number of events.

The scenario contributions enter constant drift across the full horizon. Event dates organize the inputs and rate updates; this version does not impose event-date price jumps.


In [ ]:
shock_events_df = pd.DataFrame(
    [
        ("2026-02-28", "ZHAW Daily", "oil_price_shock", 0.48),
        ("2026-06-08", "ZHAW Daily", "oil_price_shock", -0.29),
        ("2026-09-17", "ZHAW Daily", "interest_rate_shock", 0.07),
    ],
    columns=["date", "publisher", "shock_type", "price_change"],
)

shock_events_df["date"] = pd.to_datetime(shock_events_df["date"])


## Load relevant data

We load stock prices, interest rates, and WTI spot prices from `DATA_DIR`, keeping observations between `DATA_START_DATE` and `DATA_END_DATE`. The helper below reads JSON files.


In [ ]:
def load_json(relative_path):
    with (DATA_DIR / relative_path).open(encoding="utf-8") as file:
        return json.load(file)


### Stock prices and daily returns

`TICKER` selects the stock-price file. We keep the date and closing price, filter to the historical window, and sort by date. `source_close` preserves the price supplied in the file; `close` is the working price used for returns and calibration.

For **NOG**, if the source explicitly reports unadjusted prices, we multiply prices dated **before 21 September 2020 by 10** to account for its 1-for-10 reverse split. The source JSON and company-information file are not changed, and already split-adjusted input is not adjusted again. See the [SEC filing](https://www.sec.gov/Archives/edgar/data/1104485/000110448520000179/nog-20200918.htm).

All stock sources must explicitly identify their split-adjustment basis. Unadjusted non-NOG sources are rejected until a documented correction is supplied; this prevents split jumps from entering regression and volatility. The retained window must contain unique calendar dates and finite positive prices.

We then calculate:

- `daily_simple_return = close / previous_close - 1`.
- `daily_log_return = ln(close / previous_close)`, retained as a descriptive return column. Calibration later recomputes returns on a shared NYSE session grid.

The first retained return is missing because returns use only prices inside the historical window. These are price returns; dividends are not included.


In [ ]:
def prepare_stock_prices(price_data, ticker, historical_start, historical_end):
    """Validate the retained price window and apply only a documented split correction."""
    if price_data.get("ticker") != ticker:
        raise ValueError("The stock-price source ticker must match TICKER.")
    start, end = pd.Timestamp(historical_start), pd.Timestamp(historical_end)
    if pd.isna(start) or pd.isna(end) or start.tz is not None or end.tz is not None or start > end:
        raise ValueError("Historical price dates must be timezone-free and ordered.")
    prices = pd.DataFrame(price_data["data"])[["date", "close"]].copy()
    prices["date"] = pd.to_datetime(prices["date"])
    if (prices["date"].isna().any() or prices["date"].dt.tz is not None
            or not prices["date"].eq(prices["date"].dt.normalize()).all()):
        raise ValueError("Stock observation dates must be timezone-free calendar dates.")
    prices = prices.loc[prices["date"].between(start, end, inclusive="both")].sort_values("date")
    if prices.empty or prices["date"].duplicated().any():
        raise ValueError("The historical price window must contain unique observations.")
    prices["close"] = pd.to_numeric(prices["close"], errors="coerce")
    prices["source_close"] = prices["close"]
    split_adjusted = price_data.get("adjusted_for_splits")
    if not isinstance(split_adjusted, bool):
        raise ValueError("Every stock source must identify whether prices are split-adjusted.")
    if not split_adjusted:
        if ticker != "NOG":
            raise ValueError("Use split-adjusted prices; no split correction is configured for this ticker.")
        prices.loc[prices["date"] < pd.Timestamp("2020-09-21"), "close"] *= 10.0
    if not np.isfinite(prices["close"]).all() or (prices["close"] <= 0).any():
        raise ValueError("Stock log returns require finite, positive closing prices.")
    prices["daily_simple_return"] = prices["close"].pct_change(fill_method=None)
    prices["daily_log_return"] = np.log(prices["close"]).diff()
    return prices.reset_index(drop=True)


stock_prices_data = load_json(f"prices/{TICKER}.json")
stock_prices_df = prepare_stock_prices(stock_prices_data, TICKER, DATA_START_DATE, DATA_END_DATE)


### Interest rate information

We read `rates/us_interest_rates.csv` into `interest_rates_df`. The effective federal funds rate is stored in `interest_rate_percent`, in percent. We convert the dates and rates, remove missing observations, filter to the historical window, and sort by date.


In [ ]:
interest_rates_df = pd.read_csv(DATA_DIR / "rates" / "us_interest_rates.csv")[[
    "date", "effective_federal_funds_rate_percent",
]].rename(columns={"effective_federal_funds_rate_percent": "interest_rate_percent"})
interest_rates_df["date"] = pd.to_datetime(interest_rates_df["date"])
interest_rates_df["interest_rate_percent"] = pd.to_numeric(
    interest_rates_df["interest_rate_percent"], errors="coerce"
)
interest_rates_df = (
    interest_rates_df.dropna(subset=["date", "interest_rate_percent"])
    .loc[lambda data: data["date"].between(DATA_START_DATE, DATA_END_DATE, inclusive="both")]
    .sort_values("date")
    .reset_index(drop=True)
)


### WTI spot prices

We read `prices/WTI.csv` into a separate DataFrame named `wti_prices_df`. The price column is renamed to `spot_price`; `observation_date` keeps its original name and is converted to a date. We filter to the historical window and sort by date, retaining any missing spot-price observations.

In [ ]:
wti_prices_df = pd.read_csv(DATA_DIR / "prices" / "WTI.csv").rename(
    columns={"DCOILWTICO": "spot_price"}
)
wti_prices_df["observation_date"] = pd.to_datetime(wti_prices_df["observation_date"])
wti_prices_df = (
    wti_prices_df.loc[
        wti_prices_df["observation_date"].between(DATA_START_DATE, DATA_END_DATE, inclusive="both")
    ]
    .sort_values("observation_date")
    .reset_index(drop=True)
)


## Initialize synthetic price paths

Create `paths_df` with `N_PATHS` rows and one column, `day_0`. Every synthetic path starts at the latest available closing price before `SIMULATION_START_DATE` for `TICKER`.

With the current settings, the simulation starts on **1 January 2026**, and `day_0` is NOG's **$21.47 close on 31 December 2025**. The later simulation section adds one column per trading day. Rerunning this initialization resets the dataframe to its starting prices.


In [ ]:
reference_prices = stock_prices_df.loc[
    stock_prices_df["date"] < SIMULATION_START_DATE, ["date", "close"]
].sort_values("date")
if reference_prices.empty:
    raise ValueError(
        f"No closing price for {TICKER} before {SIMULATION_START_DATE:%Y-%m-%d} "
        "in the configured historical data window."
    )

initial_price_date = reference_prices.iloc[-1]["date"]
initial_share_price = float(reference_prices.iloc[-1]["close"])
paths_df = pd.DataFrame({"day_0": initial_share_price}, index=range(N_PATHS))

print(
    f"{TICKER} day_0 for simulation starting {SIMULATION_START_DATE:%d.%m.%Y}: "
    f"${initial_share_price:.2f} (close on {initial_price_date:%d.%m.%Y})"
)
paths_df


## Align the data and prepare regression inputs

Put stock, WTI, and interest-rate **levels** on the same NYSE historical session grid, then calculate all returns and changes between the same two adjacent sessions. Do not forward-fill or bridge missing observations. A missing or nonpositive WTI endpoint makes the interval unavailable for regression; it does not imply a zero oil move.

The response is the stock log return. The two inputs are:

- **Oil-shock log return:** `ln(WTI_today / WTI_previous_session)` only when `5% < abs(WTI_simple_return) < 100%`; zero for other observed, valid oil moves.
- **Interest-rate change:** current minus previous-session rate, in **percentage points**, without a magnitude threshold.

Fit on **all complete intervals**, including days when both inputs are zero. Ordinary days help estimate background growth, while the oil coefficient still uses only qualifying moves. Incomplete intervals are counted explicitly. The first session has no prior observation inside the retained historical window.


In [ ]:
market_calendar = mcal.get_calendar("NYSE")
historical_trading_dates = market_calendar.valid_days(
    start_date=DATA_START_DATE, end_date=DATA_END_DATE
).tz_localize(None)


def prepare_regression_data(stock_levels, oil_levels, rate_levels, session_dates):
    """Align levels before changes; retain all complete intervals, including ordinary days."""
    dates = pd.DatetimeIndex(session_dates)
    if dates.empty or dates.hasnans or not dates.is_unique or not dates.is_monotonic_increasing:
        raise ValueError("Historical sessions must be nonempty, unique, and ordered.")
    aligned = pd.DataFrame(index=dates)
    for frame, date_column, value_column, output_column in [
        (stock_levels, "date", "close", "stock_close"),
        (oil_levels, "observation_date", "spot_price", "wti_close"),
        (rate_levels, "date", "interest_rate_percent", "interest_rate_percent"),
    ]:
        levels = frame[[date_column, value_column]].copy()
        levels[date_column] = pd.to_datetime(levels[date_column])
        if levels[date_column].isna().any() or levels[date_column].duplicated().any():
            raise ValueError(f"{output_column} requires nonmissing, unique observation dates.")
        values = pd.to_numeric(levels.set_index(date_column)[value_column], errors="coerce")
        aligned[output_column] = values.where(np.isfinite(values)).reindex(dates)

    # Mask invalid endpoints in place before differencing; never bridge a missing session.
    valid_stock = aligned["stock_close"].where(aligned["stock_close"] > 0)
    valid_oil = aligned["wti_close"].where(aligned["wti_close"] > 0)
    aligned["previous_date"] = aligned.index.to_series().shift(1)
    aligned["stock_log_return"] = np.log(valid_stock).diff()
    aligned["wti_log_return"] = np.log(valid_oil).diff()
    aligned["wti_simple_return"] = valid_oil.pct_change(fill_method=None)
    aligned["rate_change_pp"] = aligned["interest_rate_percent"].diff()
    # Round only for threshold comparisons, so an exact 5% move is not admitted by float noise.
    absolute_oil_move = aligned["wti_simple_return"].abs().round(12)
    relevant_oil = absolute_oil_move.gt(0.05) & absolute_oil_move.lt(1.0)
    aligned["oil_shock_log_return"] = aligned["wti_log_return"].where(relevant_oil, 0.0)
    aligned.loc[~np.isfinite(aligned["wti_log_return"]), "oil_shock_log_return"] = np.nan
    aligned["complete_interval"] = np.isfinite(
        aligned[["stock_log_return", "oil_shock_log_return", "rate_change_pp"]]
    ).all(axis=1)
    history = aligned.rename_axis("date").reset_index()
    complete = history.loc[history["complete_interval"]].copy().reset_index(drop=True)
    return history, complete


aligned_history_df, regression_data_df = prepare_regression_data(
    stock_prices_df, wti_prices_df, interest_rates_df, historical_trading_dates
)
regression_excluded_intervals = int((~aligned_history_df["complete_interval"]).sum())
regression_event_rows = (
    regression_data_df["oil_shock_log_return"].ne(0)
    | regression_data_df["rate_change_pp"].ne(0)
)
print(f"Complete trading-day intervals: {len(regression_data_df):,}; excluded: {regression_excluded_intervals:,}")
print(f"Event intervals: {regression_event_rows.sum():,}; ordinary intervals: {(~regression_event_rows).sum():,}")
print(f"Qualifying oil moves: {regression_data_df['oil_shock_log_return'].ne(0).sum():,}; rate changes: {regression_data_df['rate_change_pp'].ne(0).sum():,}")


## Fit background growth and shock effects jointly

`stock_log_return = intercept + oil_beta × oil_shock_log_return + rate_beta × rate_change_pp + error`

Estimate one intercept and two coefficients by ordinary least squares on all complete intervals. The intercept estimates the stock's daily background log return when both model inputs are zero. Each slope controls for the other input. Stock and oil log returns are decimals; rate changes are in percentage points.

Ordinary days contribute to calibration even though small oil moves do not activate the oil-shock input. Each simulation session receives the background intercept exactly once; scenario events add only their coefficient-weighted inputs. These estimates describe historical associations. In-sample R² and residual volatility are diagnostics, not evidence of forecast accuracy.


In [ ]:
regression_inputs = regression_data_df[["oil_shock_log_return", "rate_change_pp"]].to_numpy(dtype=float)
stock_log_returns = regression_data_df["stock_log_return"].to_numpy(dtype=float)
design_matrix = np.column_stack([np.ones(len(regression_inputs)), regression_inputs])
if len(regression_data_df) < 4 or np.linalg.matrix_rank(design_matrix) < 3:
    raise ValueError("The two-input regression needs at least four observations and a full-rank design.")

regression_coefficients, _, _, _ = np.linalg.lstsq(design_matrix, stock_log_returns, rcond=None)
regression_intercept, oil_beta, rate_beta = map(float, regression_coefficients)


def predict_stock_log_return(oil_shock_log_return=0.0, rate_change_pp=0.0):
    """Predict a decimal stock log return from oil log return and rate change in pp."""
    oil_input = np.asarray(oil_shock_log_return, dtype=float)
    rate_input = np.asarray(rate_change_pp, dtype=float)
    if not np.isfinite(oil_input).all() or not np.isfinite(rate_input).all():
        raise ValueError("Regression inputs must be finite.")
    return regression_intercept + oil_beta * oil_input + rate_beta * rate_input


regression_data_df["predicted_stock_log_return"] = predict_stock_log_return(
    regression_data_df["oil_shock_log_return"], regression_data_df["rate_change_pp"]
)
regression_residuals = stock_log_returns - regression_data_df["predicted_stock_log_return"].to_numpy()
regression_total_variation = np.sum((stock_log_returns - stock_log_returns.mean()) ** 2)
regression_r_squared = (
    float(1 - np.sum(regression_residuals ** 2) / regression_total_variation)
    if regression_total_variation > 0 else np.nan
)
regression_residual_volatility = float(np.sqrt(np.sum(regression_residuals ** 2) / (len(stock_log_returns) - 3)))
regression_coefficients_df = pd.DataFrame({
    "parameter": ["intercept", "oil_beta", "rate_beta"],
    "coefficient": regression_coefficients,
    "input_units": ["background daily log return", "oil log return", "rate change in percentage points"],
}).set_index("parameter")
print(f"Stock log return = {regression_intercept:+.6f} {oil_beta:+.6f} × oil log return {rate_beta:+.6f} × rate change (pp)")
print(f"R²: {regression_r_squared:.4f}; complete trading-day intervals: {len(regression_data_df):,}")
regression_coefficients_df


### Visualize both coefficients of the same model

Each panel subtracts the other input's fitted contribution from observed stock log returns. The panels display the oil and interest-rate coefficients from **one combined regression**, using all complete intervals, including ordinary days. The fitted line includes the common intercept.

Return axes show log returns multiplied by 100; rate changes remain in percentage points.


In [ ]:
regression_fig, regression_axes = plt.subplots(1, 2, figsize=(13, 5.4), dpi=150)
oil_x = regression_data_df["oil_shock_log_return"].to_numpy()
rate_x = regression_data_df["rate_change_pp"].to_numpy()
oil_adjusted_y = stock_log_returns - rate_beta * rate_x
rate_adjusted_y = stock_log_returns - oil_beta * oil_x

for ax, x, y, coefficient, title, xlabel in [
    (regression_axes[0], oil_x, oil_adjusted_y, oil_beta, "Oil input", "Oil-shock log return (%)"),
    (regression_axes[1], rate_x, rate_adjusted_y, rate_beta, "Interest-rate input", "Rate change (percentage points)"),
]:
    grid = np.linspace(x.min(), x.max(), 200)
    ax.axhline(0, color="#94a3b8", linewidth=0.8)
    ax.axvline(0, color="#94a3b8", linewidth=0.8)
    ax.scatter(x, y, s=22, color="#285f87", alpha=0.35, edgecolors="white", linewidths=0.4)
    ax.plot(grid, regression_intercept + coefficient * grid, color="#ce7429", linewidth=2.2)
    ax.set_title(title, loc="left", weight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Stock log return, other contribution removed (%)")
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.16)
    ax.set_axisbelow(True)
regression_axes[0].xaxis.set_major_formatter(PercentFormatter(xmax=1.0))
regression_axes[1].xaxis.set_major_formatter("{x:.2f}")
regression_fig.suptitle(
    f"{TICKER}: daily regression with shock inputs  |  {len(regression_data_df):,} dates  |  R² = {regression_r_squared:.3f}",
    fontsize=15, weight="bold",
)
regression_fig.tight_layout()
plt.show()


## Historical volatility and simulation calendar

Use the sample standard deviation of all finite, split-corrected **stock** log returns on the historical NYSE grid. Stock volatility includes dates without complete WTI/rate inputs; only the regression needs all three series.

The simulation retains **total historical stock volatility** as a broad scenario assumption. The regression's residual volatility is displayed as a diagnostic, but it is not substituted into the paths. A model conditional on fully specified future factor paths could instead use residual noise; this notebook does not simulate those factor paths or estimate coefficient uncertainty.

Daily volatility stays in trading-day units. The simulation uses actual NYSE sessions, excluding weekends and holidays.


In [ ]:
trading_dates = market_calendar.valid_days(
    start_date=SIMULATION_START_DATE,
    end_date=SIMULATION_END_DATE - pd.Timedelta(days=1),
).tz_localize(None)
if trading_dates.empty:
    raise ValueError("The simulation period must contain at least one trading session.")
TRADING_DAYS = len(trading_dates)
price_columns = [f"day_{day}" for day in range(TRADING_DAYS + 1)]
daily_price_dates = pd.Series(
    pd.DatetimeIndex([initial_price_date, *trading_dates]),
    index=pd.Index(price_columns, name="column"), name="date",
)

historical_log_returns = aligned_history_df.loc[
    np.isfinite(aligned_history_df["stock_log_return"]), "stock_log_return"
].to_numpy(dtype=float)
if historical_log_returns.size < 2:
    raise ValueError("Volatility requires at least two finite historical stock log returns.")
daily_volatility = float(historical_log_returns.std(ddof=1))
print(f"NYSE simulation sessions: {TRADING_DAYS}; historical stock returns: {len(historical_log_returns):,}")
print(f"Daily volatility used for paths: {daily_volatility:.4%}")
print(f"Regression residual volatility (diagnostic): {regression_residual_volatility:.4%}")


### Historical stock-return distribution

The histogram shows all finite daily **log returns** of the stock in the historical data window (2020–2025 with the current settings), using the same split-corrected returns as the volatility estimate. Each observation represents one historical trading-day return, `ln(close / previous_close)`.

The horizontal axis displays log returns multiplied by 100; the vertical axis counts observations. Bin widths use the Freedman–Diaconis rule, and the full observed return range is included. The orange line marks the historical mean. This is the historical daily distribution; simulated full-period returns are a different quantity.


In [ ]:
historical_return_mean = float(historical_log_returns.mean())

returns_histogram_fig, returns_histogram_ax = plt.subplots(figsize=(11, 5.8), dpi=150)
return_histogram_counts, return_histogram_bins, _ = returns_histogram_ax.hist(
    historical_log_returns, bins="fd", color="#285f87", alpha=0.85,
    edgecolor="white", linewidth=0.5,
)
returns_histogram_ax.axvline(0, color="#64748b", linewidth=1, linestyle=":")
returns_histogram_ax.axvline(
    historical_return_mean, color="#ce7429", linewidth=2,
    label=f"Historical mean: {historical_return_mean:.3%}",
)
returns_histogram_ax.set_title(
    f"{TICKER}: historical daily log returns", loc="left", fontsize=16, weight="bold", pad=36,
)
returns_histogram_ax.text(
    0, 1.025,
    f"{DATA_START_DATE:%Y-%m-%d} to {DATA_END_DATE:%Y-%m-%d}  |  "
    f"{len(historical_log_returns):,} observations  |  Daily volatility: {daily_volatility:.2%}",
    transform=returns_histogram_ax.transAxes, fontsize=10, color="#475569",
)
returns_histogram_ax.set_xlabel("Daily stock log return (%)")
returns_histogram_ax.set_ylabel("Number of trading-day observations")
returns_histogram_ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0))
returns_histogram_ax.set_ylim(bottom=0)
returns_histogram_ax.spines[["top", "right"]].set_visible(False)
returns_histogram_ax.grid(axis="y", alpha=0.16)
returns_histogram_ax.set_axisbelow(True)
returns_histogram_ax.legend(loc="upper left", frameon=False)
returns_histogram_fig.tight_layout()
plt.show()


## Prepare the dated shock scenario

Map each event to the next simulation session. Convert oil changes with `log1p`, combine oil events on the same session, and apply the strict 5%–100% filter to their net daily move. Convert relative rate changes using the latest rate before the simulation, updating that reference after each applied rate event. Several rate events on one session add their percentage-point changes together. The audit marks oil events whose combined session move is filtered out.

The audit records original and applied dates, the converted inputs, reference rates, and skipped events. It flags individual and combined session inputs outside the historical regression range. These flags indicate extrapolation; values are not clipped. The session table contains zero factor inputs on ordinary days. No event receives an extra regression intercept.


In [ ]:
def build_scenario_inputs(events, rate_history, dates, simulation_start, simulation_end):
    """Create session inputs and an event audit; compound applied relative rate shocks."""
    start, end = pd.Timestamp(simulation_start), pd.Timestamp(simulation_end)
    dates = pd.DatetimeIndex(dates)
    if (pd.isna(start) or pd.isna(end) or start >= end or dates.empty or dates.hasnans
            or dates.tz is not None or not dates.is_unique or not dates.is_monotonic_increasing
            or not ((dates >= start) & (dates < end)).all()):
        raise ValueError("Provide ordered, unique simulation sessions inside the simulation window.")
    selected = events.copy()
    selected["date"] = pd.to_datetime(selected["date"])
    if (selected["date"].isna().any() or selected["date"].dt.tz is not None
            or not selected["date"].eq(selected["date"].dt.normalize()).all()):
        raise ValueError("Event dates must be timezone-free calendar dates.")
    selected = selected.sort_values("date", kind="stable")
    history = rate_history.copy()
    history["date"] = pd.to_datetime(history["date"])
    if (history["date"].isna().any() or history["date"].dt.tz is not None
            or not history["date"].eq(history["date"].dt.normalize()).all()):
        raise ValueError("Reference-rate dates must be timezone-free calendar dates.")
    history = history.loc[history["date"] < start].sort_values("date")
    if history["date"].duplicated().any():
        raise ValueError("Reference-rate dates must be unique before the simulation.")
    reference_rate = float(history.iloc[-1]["interest_rate_percent"]) if not history.empty else np.nan
    inputs = pd.DataFrame(0.0, index=dates, columns=["oil_shock_log_return", "rate_change_pp"])
    inputs.index.name = "date"
    rows = []
    for event in selected.itertuples(index=False):
        row = {
            "event_date": event.date, "applied_date": pd.NaT, "shock_type": event.shock_type,
            "price_change": event.price_change, "oil_shock_log_return": 0.0, "rate_change_pp": 0.0,
            "reference_rate_before_percent": reference_rate,
            "reference_rate_after_percent": reference_rate, "status": "outside_simulation_window",
        }
        if start <= event.date < end:
            session = dates.searchsorted(event.date)
            if session == len(dates):
                row["status"] = "no_remaining_session"
            else:
                change = float(event.price_change)
                if not np.isfinite(change):
                    raise ValueError("Applied shock changes must be finite.")
                if event.shock_type == "oil_price_shock":
                    if change <= -1:
                        raise ValueError("Oil price changes must be greater than -100% for log returns.")
                    row["oil_shock_log_return"] = float(np.log1p(change))
                elif event.shock_type == "interest_rate_shock":
                    if not np.isfinite(reference_rate) or reference_rate < 0 or change < -1:
                        raise ValueError("Rate events require a finite nonnegative reference and changes >= -100%.")
                    row["rate_change_pp"] = reference_rate * change
                    reference_rate += row["rate_change_pp"]
                    if not np.isfinite(reference_rate):
                        raise ValueError("The updated reference rate must be finite.")
                    row["reference_rate_after_percent"] = reference_rate
                else:
                    raise ValueError("Only oil-price and interest-rate shocks are supported.")
                applied_date = dates[session]
                for feature in inputs.columns:
                    inputs.loc[applied_date, feature] += row[feature]
                row["applied_date"] = applied_date
                row["status"] = "applied" if row["oil_shock_log_return"] or row["rate_change_pp"] else "zero_change"
        rows.append(row)
    if not np.isfinite(inputs.to_numpy()).all():
        raise ValueError("Combined scenario inputs must remain finite.")
    audit = pd.DataFrame(rows, columns=[
        "event_date", "applied_date", "shock_type", "price_change", "oil_shock_log_return",
        "rate_change_pp", "reference_rate_before_percent", "reference_rate_after_percent", "status",
    ])
    # Match the historical feature: gate the net session move, not individual intraday events.
    with np.errstate(over="raise", invalid="raise"):
        net_oil_moves = np.expm1(inputs["oil_shock_log_return"])
    absolute_net_moves = net_oil_moves.abs().round(12)
    relevant_sessions = absolute_net_moves.gt(0.05) & absolute_net_moves.lt(1.0)
    inputs["oil_shock_log_return"] = inputs["oil_shock_log_return"].where(relevant_sessions, 0.0)
    inputs["net_oil_simple_return"] = net_oil_moves
    audit["session_oil_shock_log_return"] = audit["applied_date"].map(inputs["oil_shock_log_return"])
    filtered_oil_events = (
        audit["status"].eq("applied") & audit["shock_type"].eq("oil_price_shock")
        & ~audit["applied_date"].isin(inputs.index[relevant_sessions])
    )
    audit.loc[filtered_oil_events, "status"] = "filtered_by_session_oil_threshold"
    return inputs, audit


scenario_inputs_df, scenario_event_audit_df = build_scenario_inputs(
    shock_events_df, interest_rates_df, trading_dates, SIMULATION_START_DATE, SIMULATION_END_DATE
)
scenario_inputs_df["outside_historical_range"] = False
scenario_event_audit_df["outside_historical_range"] = False
for feature in ["oil_shock_log_return", "rate_change_pp"]:
    lower, upper = regression_data_df[feature].agg(["min", "max"])
    scenario_inputs_df["outside_historical_range"] |= ~scenario_inputs_df[feature].between(lower, upper)
    scenario_event_audit_df["outside_historical_range"] |= (
        scenario_event_audit_df["status"].eq("applied")
        & ~scenario_event_audit_df[feature].between(lower, upper)
    )
scenario_event_audit_df["session_outside_historical_range"] = scenario_event_audit_df["applied_date"].isin(
    scenario_inputs_df.index[scenario_inputs_df["outside_historical_range"]]
)
print(f"Applied events: {scenario_event_audit_df['status'].eq('applied').sum()}; extrapolated sessions: {scenario_inputs_df['outside_historical_range'].sum()}")
scenario_event_audit_df


## Convert predicted log returns to the terminal median and mean

Every simulation session receives the fitted daily intercept. Each event contributes only its oil/rate slope effect, once:

`L = N * intercept + oil_beta * sum(oil_inputs) + rate_beta * sum(rate_changes_pp)`

With independent Gaussian daily noise of standard deviation `sigma`, cumulative log return has mean `L` and variance `N * sigma**2`. Consequently:

- `median_end_price = initial_share_price * exp(L)`.
- `expected_end_price = initial_share_price * exp(L + 0.5 * N * sigma**2)`.
- `daily_log_drift = L / N`.
- `daily_drift = daily_log_drift + 0.5 * sigma**2`.

This conversion preserves the regression's predicted cumulative log return in the simulated paths. The mean exceeds the median when volatility is positive. Even with no scenario events, the model includes ordinary background growth and uncertainty. These statements hold under the notebook's Gaussian-noise assumption; they do not establish an unbiased real-world price forecast. See the [GBM reference](https://www.columbia.edu/~ks20/FE-Notes/4700-07-Notes-GBM.pdf).

The combined scenario effects enter constant drift across the entire year. There are no additional dated jumps and no forced terminal prices.


In [ ]:
scenario_inputs_df["background_log_return"] = regression_intercept
scenario_inputs_df["oil_log_contribution"] = oil_beta * scenario_inputs_df["oil_shock_log_return"]
scenario_inputs_df["rate_log_contribution"] = rate_beta * scenario_inputs_df["rate_change_pp"]
scenario_inputs_df["predicted_stock_log_return"] = predict_stock_log_return(
    scenario_inputs_df["oil_shock_log_return"], scenario_inputs_df["rate_change_pp"]
)
background_horizon_log_return = TRADING_DAYS * regression_intercept
oil_horizon_log_return = float(scenario_inputs_df["oil_log_contribution"].sum())
rate_horizon_log_return = float(scenario_inputs_df["rate_log_contribution"].sum())
horizon_log_return = float(scenario_inputs_df["predicted_stock_log_return"].sum())
horizon_log_variance = TRADING_DAYS * daily_volatility ** 2
with np.errstate(over="raise", invalid="raise"):
    median_end_price = float(initial_share_price * np.exp(horizon_log_return))
    expected_end_price = float(initial_share_price * np.exp(horizon_log_return + 0.5 * horizon_log_variance))
if not np.isfinite([median_end_price, expected_end_price]).all() or min(median_end_price, expected_end_price) <= 0:
    raise ValueError("The scenario requires representable positive mean and median endpoint prices.")
daily_log_drift = horizon_log_return / TRADING_DAYS
daily_drift = daily_log_drift + 0.5 * daily_volatility ** 2

simulation_parameters_df = pd.DataFrame([{
    "start_price": initial_share_price, "median_end_price": median_end_price,
    "expected_end_price": expected_end_price, "trading_days": TRADING_DAYS,
    "regression_observations": len(regression_data_df), "historical_returns": len(historical_log_returns),
    "background_horizon_log_return": background_horizon_log_return,
    "oil_horizon_log_return": oil_horizon_log_return,
    "rate_horizon_log_return": rate_horizon_log_return,
    "horizon_log_return": horizon_log_return, "daily_volatility": daily_volatility,
    "regression_residual_volatility": regression_residual_volatility,
    "daily_drift": daily_drift, "daily_log_drift": daily_log_drift, "random_seed": RANDOM_SEED,
}])
print(f"Starting price: ${initial_share_price:.2f}")
print(f"Horizon log return: {horizon_log_return:.4%} (background {background_horizon_log_return:.4%}, oil {oil_horizon_log_return:.4%}, rates {rate_horizon_log_return:.4%})")
print(f"Theoretical median endpoint: ${median_end_price:.4f}; expected mean endpoint: ${expected_end_price:.4f}")
print(f"Daily GBM drift: {daily_drift:.6%}; mean daily log increment: {daily_log_drift:.6%}")
simulation_parameters_df


## Generate geometric Brownian-motion paths

For each path and trading day, draw an independent standard normal value `Z` and update:

`price_today = price_yesterday * exp(daily_drift - 0.5 * daily_volatility**2 + daily_volatility * Z)`

This gives `E[final_price] = expected_end_price`. Individual final prices vary; neither the last value nor the sample mean is forced to equal the target. The fixed seed makes results reproducible. Rerunning this section replaces the generated daily columns while preserving `day_0` and the path index.

The model uses constant historical volatility and independent normal log-return increments. It does not model default or volatility clustering.


In [ ]:
def generate_gbm_paths(start_prices, dates, target_price, volatility, seed=2026):
    """Generate GBM prices with a specified expected terminal price and daily volatility."""
    starts = pd.Series(start_prices, copy=True, dtype=float)
    dates = pd.DatetimeIndex(dates)
    target_price, volatility = float(target_price), float(volatility)
    if starts.empty or not starts.index.is_unique or not np.isfinite(starts).all() or (starts <= 0).any():
        raise ValueError("Starting prices must be finite and positive, with a unique path index.")
    if dates.empty or dates.hasnans or not dates.is_unique or not dates.is_monotonic_increasing:
        raise ValueError("Trading dates must be nonempty, unique, and ordered.")
    if not np.isfinite(target_price) or target_price <= 0 or not np.isfinite(volatility) or volatility < 0:
        raise ValueError("The target must be finite and positive; daily volatility must be finite and nonnegative.")

    n_days = len(dates)
    start_values = starts.to_numpy()
    # Subtract logs instead of forming a ratio that can overflow for valid prices.
    start_logs = np.log(start_values)
    path_daily_drift = (np.log(target_price) - start_logs) / n_days
    rng = np.random.default_rng(seed)
    normal_increments = rng.standard_normal((len(starts), n_days))
    log_increments = path_daily_drift[:, None] - 0.5 * volatility ** 2 + volatility * normal_increments
    cumulative_log_returns = np.column_stack([
        np.zeros(len(starts)), np.cumsum(log_increments, axis=1)
    ])
    with np.errstate(over="raise", invalid="raise"):
        values = np.exp(start_logs[:, None] + cumulative_log_returns)
    values[:, 0] = start_values  # Preserve the exact supplied day_0 values.
    if not np.isfinite(values).all() or (values <= 0).any():
        raise ValueError("The GBM simulation produced a nonfinite or nonpositive price.")
    return pd.DataFrame(
        values, index=starts.index, columns=[f"day_{day}" for day in range(n_days + 1)]
    )


paths_df = generate_gbm_paths(
    paths_df["day_0"], trading_dates, expected_end_price, daily_volatility, seed=RANDOM_SEED
)
terminal_prices = paths_df.iloc[:, -1]
print(f"Generated {len(paths_df):,} paths: day_0 through day_{TRADING_DAYS}.")
print(f"Theoretical expected end price: ${expected_end_price:.4f}; theoretical median: ${median_end_price:.4f}")
print(f"Simulated mean end price: ${terminal_prices.mean():.4f}")
print(f"Simulated median end price: ${terminal_prices.median():.4f}")


### Visualize the GBM paths

Faint lines show 100 example paths. The band contains the middle 90% of all simulated prices on each date. The mean, median, and theoretical expected-price curve show how the paths evolve toward the **expected** terminal target. The scenario's shock effects are incorporated in drift, so the chart does not add event-date jumps.


In [ ]:
path_values = paths_df.to_numpy(dtype=float)
path_dates = pd.DatetimeIndex(daily_price_dates.loc[paths_df.columns])
path_low, path_median, path_high = np.quantile(path_values, [0.05, 0.50, 0.95], axis=0)
path_mean = path_values.mean(axis=0)
theoretical_expected_prices = initial_share_price * np.exp(daily_drift * np.arange(TRADING_DAYS + 1))
chart_rng = np.random.default_rng(7)
example_rows = chart_rng.choice(len(paths_df), size=min(100, len(paths_df)), replace=False)

paths_fig, paths_ax = plt.subplots(figsize=(12, 6.5), dpi=150)
paths_ax.fill_between(path_dates, path_low, path_high, color="#dbeafe", alpha=0.7, label="5th–95th percentiles")
paths_ax.plot(path_dates, path_values[example_rows].T, color="#7095b8", alpha=0.12, linewidth=0.7)
paths_ax.plot(path_dates, path_median, color="#164e78", linewidth=2, label="Median")
paths_ax.plot(path_dates, path_mean, color="#c06b27", linewidth=1.7, label="Simulated mean")
paths_ax.plot(path_dates, theoretical_expected_prices, color="#111827", linestyle="--", linewidth=1.5, label="Theoretical mean")
paths_ax.scatter(path_dates[-1], expected_end_price, s=45, color="#111827", zorder=5, clip_on=False)
paths_ax.annotate(
    f"Expected mean: ${expected_end_price:.2f}\nTheoretical median: ${median_end_price:.2f}", xy=(path_dates[-1], expected_end_price),
    xytext=(-10, 16), textcoords="offset points", ha="right", fontsize=10,
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.9, "pad": 3},
)
paths_ax.set_title(f"{TICKER}: {len(paths_df):,} geometric Brownian-motion paths", loc="left", fontsize=16, weight="bold", pad=44)
paths_ax.set_xlabel("Trading date")
paths_ax.set_ylabel("Share price (USD)")
paths_ax.yaxis.set_major_formatter("${x:.0f}")
paths_ax.set_ylim(bottom=0)
paths_ax.set_xlim(path_dates[0], path_dates[-1])
paths_ax.spines[["top", "right"]].set_visible(False)
paths_ax.grid(alpha=0.16)
paths_ax.set_axisbelow(True)
paths_ax.legend(loc="lower left", bbox_to_anchor=(0, 1.01), ncol=4, frameon=False, fontsize=9, borderaxespad=0)
paths_fig.tight_layout()
plt.show()


## Export generated paths

`export_paths_to_parquet` saves the complete `paths_df` to **`EXPORT_ROOT/[ticker]/regression/paths.parquet`**, with `EXPORT_ROOT = Path("export")` by default. With the current settings, this contains 10,000 rows and the price columns `day_0` through `day_251`.

Each export uses the same folder, with no timestamp. A successful export replaces the previous `paths.parquet`. Parquet preserves the floating-point values, column order, dtypes, and path index. The current `RangeIndex` is stored as metadata, so it does not add a physical price column. Snappy compression is lossless.

The method uses a unique temporary file in the destination directory, reads it back and checks exact equality before replacing `paths.parquet`. If writing or verification fails, the previous file is preserved. It returns the full saved path. Only the export location and dimensions are printed.

To load the file later: `loaded_paths_df = pd.read_parquet(exported_paths_file)`.


In [ ]:
def export_paths_to_parquet(paths, ticker, trading_days, export_root="export"):
    """Save complete regression paths to a fixed folder and verify a lossless round trip."""
    if not isinstance(trading_days, (int, np.integer)) or isinstance(trading_days, bool) or trading_days < 1:
        raise ValueError("The number of trading days must be a positive integer.")
    if not isinstance(ticker, str) or not ticker.strip() or ticker in {".", ".."} or Path(ticker).name != ticker:
        raise ValueError("The ticker must be a single nonempty folder name.")
    expected_columns = [f"day_{day}" for day in range(trading_days + 1)]
    if list(paths.columns) != expected_columns:
        raise ValueError("Export requires ordered columns day_0 through the final simulation day.")
    if paths.empty or not paths.index.is_unique or not all(pd.api.types.is_numeric_dtype(dtype) for dtype in paths.dtypes):
        raise ValueError("Export requires nonempty numeric paths with a unique path index.")
    price_values = paths.to_numpy(dtype=float)
    if not np.isfinite(price_values).all() or (price_values <= 0).any():
        raise ValueError("Regression paths must contain finite, positive prices.")

    export_directory = Path(export_root).expanduser().resolve() / ticker / "regression"
    export_directory.mkdir(parents=True, exist_ok=True)
    export_file = export_directory / "paths.parquet"
    # A unique sibling file avoids collisions between concurrent notebook runs.
    with NamedTemporaryFile(dir=export_directory, prefix=".paths-", suffix=".parquet", delete=False) as staging:
        temporary_file = Path(staging.name)
    try:
        paths.to_parquet(temporary_file, engine="pyarrow", compression="snappy", index=None)
        restored_paths = pd.read_parquet(temporary_file, engine="pyarrow")
        pd.testing.assert_frame_equal(paths, restored_paths, check_exact=True)
        temporary_file.replace(export_file)
    except Exception:
        temporary_file.unlink(missing_ok=True)
        raise
    return export_file


exported_paths_file = export_paths_to_parquet(
    paths_df, TICKER, TRADING_DAYS, export_root=EXPORT_ROOT
)
print(f"Exported and verified {paths_df.shape[0]:,} paths × {paths_df.shape[1]:,} price columns.")
print(f"Parquet file: {exported_paths_file}")
